In [ ]:
import serial
import struct
import time

SERIAL_PORT = '/dev/ttyACM0' 
BAUD_RATE = 115200

# --- YARDIMCI FONKSİYONLAR (Öncekiyle aynı) ---
def send_msp_command(ser, cmd, data=[]):
    size = len(data)
    checksum = size ^ cmd
    for b in data: checksum ^= b
    header = struct.pack('<3sBB', b'$M<', size, cmd)
    ser.write(header + bytes(data) + struct.pack('<B', checksum))

def send_rc_commands(ser, roll, pitch, throttle, yaw, aux1, aux2, aux3, aux4):
    data = struct.pack('<8H', roll, pitch, throttle, yaw, aux1, aux2, aux3, aux4)
    send_msp_command(ser, 200, list(data))

def read_radio_switches(ser):
    ser.reset_input_buffer()
    send_msp_command(ser, 105)
    time.sleep(0.02)
    if ser.in_waiting >= 6:
        if ser.read(3) == b'$M>':
            size = ser.read(1)[0]
            cmd = ser.read(1)[0]
            if cmd == 105:
                payload = ser.read(size)
                checksum = ser.read(1)[0]
                if len(payload) >= 12:
                    return struct.unpack('<6H', payload[:12])
    return None

def read_altitude(ser):
    ser.reset_input_buffer()
    send_msp_command(ser, 109)
    time.sleep(0.02)
    if ser.in_waiting >= 6:
        if ser.read(3) == b'$M>':
            size = ser.read(1)[0]
            cmd = ser.read(1)[0]
            if cmd == 109:
                payload = ser.read(size)
                checksum = ser.read(1)[0]
                if len(payload) >= 6:
                    return struct.unpack('<ih', payload)[0]
    return None

# --- PID SINIFI ---
class AltitudePID:
    def __init__(self, Kp, Ki, Kd):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.prev_error = 0
        self.integral = 0
        self.last_time = time.time()
        
    def compute(self, target_altitude, current_altitude):
        current_time = time.time()
        dt = current_time - self.last_time
        if dt <= 0.0: dt = 0.01 
        
        error = target_altitude - current_altitude
        P = self.Kp * error
        self.integral += error * dt
        self.integral = max(min(self.integral, 50), -50) # Rüzgar birikmesini engelle
        I = self.Ki * self.integral
        D = self.Kd * ((error - self.prev_error) / dt)
        
        self.prev_error = error
        self.last_time = current_time
        return P + I + D

# --- ANA GÖREV DÖNGÜSÜ ---
try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0.1)
    
    # PID Katsayıları
    alt_pid = AltitudePID(Kp=1.2, Ki=0.1, Kd=0.5)
    
    # --- GÖREV PARAMETRELERİ ---
    TARGET_ALTITUDE_CM = 200  # 2 Metre!
    HOVER_DURATION = 10.0     # 2 metrede kaç saniye asılı kalacak?
    
    # Rampa (Asansör) Hızları (cm/saniye)
    CLIMB_RATE_CM_S = 40.0    # Saniyede 40 cm tırman (2 metreye 5 saniyede çıkar)
    DESCEND_RATE_CM_S = 30.0  # Saniyede 30 cm in (Daha yavaş ve güvenli)
    
    mission_state = "IDLE" 
    hover_start_time = 0
    dynamic_target = 0.0      # PID'nin takip edeceği HAREKETLİ hedef
    loop_dt = 0.05            # Döngü süresi (20Hz = 0.05 sn)

    print("\n--- OTONOM SİSTEM HAZIR ---")
    print(f"Hedef: {TARGET_ALTITUDE_CM} cm | Bekleme: {HOVER_DURATION} saniye")
    print("Kumandadan ARM şalterini kaldırdığınızda otonom kalkış başlayacaktır.")

    while True:
        loop_start = time.time()
        
        radio_data = read_radio_switches(ser)
        current_alt = read_altitude(ser)
        if current_alt is None: current_alt = 0

        if radio_data:
            aux1_arm = radio_data[4]
            aux2_emg = radio_data[5]
            
            # ACİL DURUM
            if aux2_emg > 1500:
                print("\n[!] ACİL DURUM ŞALTERİ ÇEKİLDİ! MOTORLAR SUSTURULUYOR [!]")
                break
            
            # KUMANDADAN MANUEL İPTAL
            if aux1_arm < 1500 and mission_state != "IDLE":
                print("\n[-] Pilot ARM şalterini indirdi. Görev İptal Edildi!")
                mission_state = "IDLE"
                send_rc_commands(ser, 1500, 1500, 1000, 1500, 1000, 1000, 1000, 1000)
            
            # OTONOM GÖREVİ BAŞLAT
            elif aux1_arm > 1500 and mission_state == "IDLE":
                mission_state = "ARMED_WAIT"
                print("\n[+] ARM edildi. Motorlar rölantide. 2 saniye sonra kalkış başlıyor...")
                time.sleep(2) # Pervanelerin devrini alması için kısa bir bekleme
                
                mission_state = "CLIMBING"
                dynamic_target = current_alt # Kalkışa bulunduğumuz yerden başla
                alt_pid.integral = 0
                print("\n[++] Otonom Kalkış (Yumuşak Rampa) Başladı!")

            # 1. YUMUŞAK YÜKSELİŞ (SOFT TAKEOFF)
            if mission_state == "CLIMBING":
                # Hedefi her döngüde azar azar artır
                dynamic_target += CLIMB_RATE_CM_S * loop_dt
                if dynamic_target > TARGET_ALTITUDE_CM:
                    dynamic_target = TARGET_ALTITUDE_CM
                
                # PID hareketli hedefi takip eder
                pid_out = alt_pid.compute(dynamic_target, current_alt)
                throttle_cmd = int(1500 + pid_out)
                safe_throttle = max(1400, min(throttle_cmd, 1650))
                
                send_rc_commands(ser, 1500, 1500, safe_throttle, 1500, 2000, 2000, 1000, 1000)
                print(f"[YÜKSELİYOR] Hedef: {int(dynamic_target)}cm | Gerçek: {current_alt}cm | Motor: {safe_throttle}")
                
                # Asıl hedefe vardık mı?
                if dynamic_target >= TARGET_ALTITUDE_CM and abs(TARGET_ALTITUDE_CM - current_alt) <= 15:
                    mission_state = "HOVERING"
                    hover_start_time = time.time()
                    print(f"\n[!] Hedefe Ulaşıldı. {HOVER_DURATION} saniye asılı kalınıyor.")

            # 2. HAVADA ASILI KALMA (HOVER)
            elif mission_state == "HOVERING":
                pid_out = alt_pid.compute(TARGET_ALTITUDE_CM, current_alt)
                throttle_cmd = int(1500 + pid_out)
                safe_throttle = max(1450, min(throttle_cmd, 1550)) # Hover sırasında motorlar çok agresif tepki vermesin
                
                send_rc_commands(ser, 1500, 1500, safe_throttle, 1500, 2000, 2000, 1000, 1000)
                
                if (time.time() - hover_start_time) >= HOVER_DURATION:
                    mission_state = "DESCENDING"
                    dynamic_target = TARGET_ALTITUDE_CM # İniş rampasını yukarıdan başlat
                    print("\n[!] Süre doldu. Yumuşak otonom iniş başlıyor...")

            # 3. YUMUŞAK İNİŞ (SOFT LANDING)
            elif mission_state == "DESCENDING":
                # Hedefi her döngüde azar azar azalt
                dynamic_target -= DESCEND_RATE_CM_S * loop_dt
                if dynamic_target < 0:
                    dynamic_target = 0
                    
                pid_out = alt_pid.compute(dynamic_target, current_alt)
                throttle_cmd = int(1500 + pid_out)
                safe_throttle = max(1350, min(throttle_cmd, 1550))
                
                send_rc_commands(ser, 1500, 1500, safe_throttle, 1500, 2000, 2000, 1000, 1000)
                print(f"[İNİYOR] Hedef: {int(dynamic_target)}cm | Gerçek: {current_alt}cm | Motor: {safe_throttle}")
                
                # Yere indik mi? (Barometre sapması yüzünden 15 cm'yi yer kabul ediyoruz)
                if current_alt <= 15 and dynamic_target <= 10:
                    mission_state = "LANDED"
                    print("\n[+] İniş Başarılı. Motorlar kapatılıyor.")

            # 4. GÖREV BİTİŞİ
            elif mission_state == "LANDED":
                send_rc_commands(ser, 1500, 1500, 1000, 1500, 2000, 1000, 1000, 1000)
                # Pilot ARM şalterini indirene kadar kod bekler.

        # Döngü zamanlamasını 20Hz'de (0.05s) tutmak için
        elapsed = time.time() - loop_start
        if elapsed < loop_dt:
            time.sleep(loop_dt - elapsed)

except KeyboardInterrupt:
    print("\n[!] KLAVYEDEN DURDURULDU [!]")
finally:
    if 'ser' in locals():
        for _ in range(5):
            send_rc_commands(ser, 1500, 1500, 1000, 1500, 1000, 1000, 1000, 1000)
            time.sleep(0.05)
        ser.close()
        print("Sistem güvenli şekilde kapatıldı.")